
###### 06_model_evaluation

###### Purpose

- Evaluate the trained telco churn models, select the best-performing model using ROC-AUC, and save its run ID and evaluation metrics in a Unity Catalog Delta table for downstream model registration.

###### Input

- Read model_training_results,  sort by best metric and get the best model info


######  Output

- write the best model info results to the table dbw_agentic_ai_dev.telco_ai.best_model_info


######  Architecture

```text

model_training_results
      ↓
Sort by roc_auc DESC
      ↓
Select best model row
      ↓
Create best_model_info
      ↓
Save Best-Model Metadata to Delta Table

```


###### Skills Covered

- Model Evaluation

- Model Comparison

- ROC-AUC

- MLflow Run Metadata

- Delta Lake

###### Section 0 : Call project config notebook

In [0]:
%run ./00_project_config

###### Section 1 :  Read Model-Training Results

In [0]:
results_df = spark.table(MODEL_RESULTS_TABLE)

###### Section 2 : Rank Candidate Models

In [0]:
display(results_df.orderBy("roc_auc", ascending=False))

###### Section 3 :  Select the Best Model

In [0]:
valid_results_df = results_df.filter("roc_auc IS NOT NULL")

best_row = (
    valid_results_df
    .orderBy("roc_auc", ascending=False)
    .limit(1)
    .collect()[0]
)

if valid_results_df.count() == 0:
    raise ValueError("No models with a valid ROC-AUC score were found.")

###### Section 4 :  Save Best-Model Metadata

In [0]:

best_model_info = [{
    "best_run_id": best_row["run_id"],
    "best_model_name": best_row["model"],
    "best_roc_auc": float(best_row["roc_auc"]),
    "best_accuracy": float(best_row["accuracy"]),
    "best_precision": float(best_row["precision"]),
    "best_recall": float(best_row["recall"]),
    "best_f1_score": float(best_row["f1_score"])
}]

best_model_df = spark.createDataFrame(best_model_info)

best_model_df.write.format("delta").mode("overwrite").saveAsTable(BEST_MODEL_TABLE)

display(best_model_df)

###### Section 5 :  Validate Saved Metadata

In [0]:
display(spark.table(BEST_MODEL_TABLE))

saved_best_model_df = spark.table(BEST_MODEL_TABLE)

assert saved_best_model_df.count() == 1
assert saved_best_model_df.first()["best_run_id"] is not None

###### Notebook Summary

- Loaded model-comparison results from the Delta table.

- Ranked candidate models using ROC-AUC as the primary metric.

- Selected the highest-performing model and captured its MLflow run ID.

- Saved the selected model's metadata and evaluation metrics to the best_model_info Delta table.

######  Key Learnings

- ROC-AUC is used as the primary selection metric because the churn classes are imbalanced and probability ranking is important for retention prioritization.

- Accuracy alone can hide poor performance on smaller or weaker classes.

- The selected MLflow run ID provides the link between model evaluation and downstream model registration.

###### Notebook Conclusion

- In this notebook, we built a model-evaluation workflow that compares candidate models using multiple performance metrics and selects the strongest model using ROC-AUC.

- The selected model’s MLflow run ID and evaluation metrics were saved to a Delta table, enabling reproducible model registration in the next phase.

- This supports data-driven model selection rather than relying on assumptions or accuracy alone.- In this notebook, we built a model evaluation workflow that compares multiple models using performance metrics and selects the best model for deployment.

- This enables data-driven model selection instead of relying on assumptions or a single evaluation metric.

- This will be used in the next notebook to explore the MLflow experiment runs, artifacts, and logged model information.

###### Next Notebook

07_mlflow_tracking

- Inspect the MLflow experiment, compare logged runs, and verify the metrics, model artifacts, signatures, and input examples associated with the selected model.